In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# OPTIMIZED UniXcoder + PrimeVul - MULTI-GPU SUPPORT
# 5 Epochs, Efficient, Multi-GPU Ready
# ============================================================

import os, json, torch, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import DataParallel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import platform, psutil
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# GPU SETUP - MULTI-GPU DETECTION
# ============================================================
print("="*60)
print("GPU CONFIGURATION")
print("="*60)
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_multi_gpu = torch.cuda.device_count() > 1

print(f"\nPrimary Device: {device}")
print(f"Multi-GPU Training: {use_multi_gpu}")
if use_multi_gpu:
    print(f"Using {torch.cuda.device_count()} GPUs")
print("="*60)

# --------------------
# DATASET PATH
# --------------------
DATASET_PATH = "/kaggle/input/primevul-xcoder1"
print("\nDataset files:", os.listdir(DATASET_PATH))

# ============================================================
# LOAD PRIMEVUL JSONL
# ============================================================
def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            data.append({
                "code": obj["func"],
                "label": int(obj["target"])
            })
    return pd.DataFrame(data)

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)
print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print("\nTrain label distribution:")
print(train_df["label"].value_counts())
print(f"Class 0: {(train_df['label']==0).sum()} ({(train_df['label']==0).sum()/len(train_df)*100:.2f}%)")
print(f"Class 1: {(train_df['label']==1).sum()} ({(train_df['label']==1).sum()/len(train_df)*100:.2f}%)")

# ============================================================
# CLASS WEIGHTS
# ============================================================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("\n" + "="*60)
print("CLASS WEIGHTS:")
print(f"Class 0: {class_weights[0]:.4f}")
print(f"Class 1: {class_weights[1]:.4f}")
print("="*60)

# ============================================================
# TOKENIZER & DATASET
# ============================================================
MODEL_NAME = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 512

class PrimeVulDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.codes[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Adjust batch size for multi-GPU
BATCH_SIZE = 8 * torch.cuda.device_count() if use_multi_gpu else 8

train_dataset = PrimeVulDataset(train_df)
val_dataset = PrimeVulDataset(val_df)
test_dataset = PrimeVulDataset(test_df)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=2,  # Parallel data loading
    pin_memory=True  # Faster data transfer to GPU
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

print(f"\nBatch size (total): {BATCH_SIZE}")
if use_multi_gpu:
    print(f"Batch size per GPU: {BATCH_SIZE // torch.cuda.device_count()}")
print(f"Training batches per epoch: {len(train_loader)}")

# ============================================================
# MODEL - MULTI-GPU READY
# ============================================================
print("\nLoading UniXcoder model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)

# Enable multi-GPU if available
if use_multi_gpu:
    print(f"Wrapping model with DataParallel for {torch.cuda.device_count()} GPUs...")
    model = DataParallel(model)
    
model = model.to(device)

# Ensure all parameters are trainable
for param in model.parameters():
    param.requires_grad = True

print(f"Model loaded on: {device}")
if use_multi_gpu:
    print(f"Model replicated across {torch.cuda.device_count()} GPUs")

# ============================================================
# TRAINING SETUP
# ============================================================
EPOCHS = 5
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.1

optimizer = AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=0.01,
    eps=1e-8
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

print("\n" + "="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Batch Size (total): {BATCH_SIZE}")
if use_multi_gpu:
    print(f"Batch Size (per GPU): {BATCH_SIZE // torch.cuda.device_count()}")
print(f"Warmup Steps: {warmup_steps}")
print(f"Total Steps: {total_steps}")
print(f"Max Sequence Length: {MAX_LEN}")
print("="*60)

# ============================================================
# EVALUATION FUNCTION
# ============================================================
def evaluate(model, data_loader, dataset_name="Validation"):
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    avg_loss = total_loss / len(data_loader)
    
    print(f"\n{dataset_name} Results:")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall: {rec:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  Predictions - Class 0: {(y_pred==0).sum()}, Class 1: {(y_pred==1).sum()}")
    
    return acc, prec, rec, f1, avg_loss, y_true, y_pred

# ============================================================
# TRAINING LOOP
# ============================================================
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

best_val_f1 = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_interval = max(len(train_loader) // 5, 1)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 60)
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        
        if (batch_idx + 1) % progress_interval == 0:
            avg_loss = total_loss / (batch_idx + 1)
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
            
            # GPU memory usage
            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    mem_allocated = torch.cuda.memory_allocated(i) / 1e9
                    mem_reserved = torch.cuda.memory_reserved(i) / 1e9
                    print(f"    GPU {i} Memory: {mem_allocated:.2f}GB allocated, {mem_reserved:.2f}GB reserved")

    avg_train_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} Complete | Avg Loss: {avg_train_loss:.4f}")
    
    # Validation
    val_acc, val_prec, val_rec, val_f1, val_loss, _, _ = evaluate(model, val_loader, "Validation")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        # Save model (handle DataParallel wrapper)
        model_to_save = model.module if use_multi_gpu else model
        torch.save(model_to_save.state_dict(), '/kaggle/working/best_unixcoder_model.pt')
        print(f"  ✓ New best model saved! (F1: {best_val_f1:.4f})")

print("\n" + "="*60)
print(f"Training Complete! Best F1: {best_val_f1:.4f} at Epoch {best_epoch}")
print("="*60)

# Load best model
if use_multi_gpu:
    model.module.load_state_dict(torch.load('/kaggle/working/best_unixcoder_model.pt'))
else:
    model.load_state_dict(torch.load('/kaggle/working/best_unixcoder_model.pt'))
print("\nLoaded best model for evaluation")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
print("\n" + "="*60)
print("FINAL TEST EVALUATION")
print("="*60)

test_acc, test_prec, test_rec, test_f1, test_loss, y_true, y_pred = evaluate(model, test_loader, "Test")

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print("\n" + "="*60)
print("DETAILED RESULTS")
print("="*60)
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn} | FP: {fp}")
print(f"  FN: {fn} | TP: {tp}")
print("="*60)

# ============================================================
# SAVE RESULTS
# ============================================================
results = {
    "dataset": "PrimeVul",
    "model": "UniXcoder",
    "training_config": {
        "epochs": EPOCHS,
        "best_epoch": best_epoch,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LEN,
        "multi_gpu": use_multi_gpu,
        "num_gpus": torch.cuda.device_count()
    },
    "metrics": {
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1_score": float(test_f1),
        "fpr": float(fpr),
        "fnr": float(fnr)
    },
    "confusion_matrix": {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    },
    "hardware": {
        "gpu_count": torch.cuda.device_count(),
        "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2)
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("/kaggle/working/UniXcoder_PrimeVul_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ Results saved!")
print("\n" + "="*60)
print("EXECUTION COMPLETE")
print("="*60)

GPU CONFIGURATION
CUDA Available: True
Number of GPUs: 2

GPU 0: Tesla T4
  Memory: 15.64 GB

GPU 1: Tesla T4
  Memory: 15.64 GB

Primary Device: cuda:0
Multi-GPU Training: True
Using 2 GPUs

Dataset files: ['primevul_test_paired.jsonl', 'primevul_train_paired.jsonl', 'primevul_valid_paired.jsonl']

DATASET STATISTICS
Train samples: 7578
Val samples: 960
Test samples: 870

Train label distribution:
label
1    3789
0    3789
Name: count, dtype: int64
Class 0: 3789 (50.00%)
Class 1: 3789 (50.00%)

CLASS WEIGHTS:
Class 0: 1.0000
Class 1: 1.0000


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wrapping model with DataParallel for 2 GPUs...
Model loaded on: cuda:0
Model replicated across 2 GPUs

TRAINING CONFIGURATION
Epochs: 5
Learning Rate: 5e-05
Batch Size (total): 16
Batch Size (per GPU): 8
Warmup Steps: 237
Total Steps: 2370
Max Sequence Length: 512

STARTING TRAINING

Epoch 1/5
------------------------------------------------------------


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

  Batch 94/474 | Loss: 0.7149 | LR: 1.98e-05
    GPU 0 Memory: 2.10GB allocated, 4.93GB reserved
    GPU 1 Memory: 0.02GB allocated, 3.18GB reserved
  Batch 188/474 | Loss: 0.7158 | LR: 3.97e-05
    GPU 0 Memory: 2.10GB allocated, 4.93GB reserved
    GPU 1 Memory: 0.02GB allocated, 3.18GB reserved
  Batch 282/474 | Loss: 0.7143 | LR: 4.89e-05
    GPU 0 Memory: 2.10GB allocated, 4.93GB reserved
    GPU 1 Memory: 0.02GB allocated, 3.18GB reserved
  Batch 376/474 | Loss: 0.7121 | LR: 4.67e-05
    GPU 0 Memory: 2.10GB allocated, 4.93GB reserved
    GPU 1 Memory: 0.02GB allocated, 3.18GB reserved
  Batch 470/474 | Loss: 0.7122 | LR: 4.45e-05
    GPU 0 Memory: 2.10GB allocated, 4.93GB reserved
    GPU 1 Memory: 0.02GB allocated, 3.18GB reserved

Epoch 1 Complete | Avg Loss: 0.7123

Validation Results:
  Loss: 0.6942
  Accuracy: 0.5000
  Precision: 0.0000
  Recall: 0.0000
  F1 Score: 0.0000
  Predictions - Class 0: 960, Class 1: 0

Epoch 2/5
---------------------------------------------------